# 1. Robust Uncertainty — tham số cầu bất định

Bước 1 của chuỗi **Robust → Multi-Objective Optimization → TOPSIS**.

- **Đọc:** `Model - Data used/` (data đã clean ở `Validation.ipynb`, `Validation_2_Network.ipynb`).
- **Tính:** cầu theo nhóm khách × tháng (pallet), tham số Bertsimas & Sim: `d̄_k` (cầu danh nghĩa) và `d̂_k` (độ lệch tối đa) — Decision_Log 5.9, 6.1.
- **Phân tích:** mức độ bất định của cầu (mùa vụ, CV) — bằng chứng cho đặc tính ngành personal care / FMCG.
- **Ghi:** `Model Outputs/demand_robust.parquet`, `Model Outputs/demand_monthly.parquet` → notebook 2 đọc.

**Ý nghĩa robust (Bertsimas & Sim):** cầu thật của nhóm k nằm trong `[d̄_k, d̄_k + d̂_k]`. Ngân sách `Γ` = số nhóm được phép lệch lên mức xấu nhất cùng lúc. `Γ = 0` → mô hình tất định; `Γ` càng lớn → nghiệm càng an toàn nhưng càng đắt. Ràng buộc robust được viết trong notebook 2; notebook này chỉ chuẩn bị `d̄`, `d̂`.

In [ ]:
# ===== SETUP =====
import pandas as pd                       # xử lý bảng
import numpy as np                        # tính toán
from pathlib import Path                  # ghép đường dẫn

pd.set_option("display.width", 200)

IN = Path("../Model - Data used")         # data đã clean (chỉ đọc)
OUT = Path("../Model Outputs")            # kết quả của các notebook model
OUT.mkdir(exist_ok=True)                  # tạo folder nếu chưa có

mp = pd.read_parquet(IN / "shipto_group_map.parquet")                     # ShipToID -> ShipToGroupID
G = sorted(mp["ShipToGroupID"].unique())                                 # tập nhóm khách (3.347)
print(f"Nhóm khách: {len(G):,}")

## 1.1 Cầu theo nhóm khách × tháng → `d̄`, `d̂`

Pallet = kg đặt ÷ `KGPerPallet` của từng sản phẩm, rồi cộng lên nhóm khách. Kỳ: tháng 1–11/2025. Tháng không đặt hàng = 0 (Decision_Log 5.9).

In [ ]:
# ===== 1.1 DEMAND: kg -> pallet -> nhóm khách × tháng -> d̄, d̂ =====
# Chỉ đọc 4 cột cần (file 24 triệu dòng)
so = pd.read_parquet(IN / "so_b2b.parquet",
                     columns=["OrderedTimestamp", "ProductID", "ShipToID", "QtyOrderedInKG"])
pm = pd.read_parquet(IN / "product_master.parquet", columns=["ProductID", "KGPerPallet"])
print(f"Dòng đơn: {len(so):,} | tấn đặt: {so['QtyOrderedInKG'].sum()/1000:,.0f}")

# (1) kg -> pallet theo từng sản phẩm
kgpp = pm.set_index("ProductID")["KGPerPallet"]                           # kg/pallet của mỗi sản phẩm
so["Pallet"] = so["QtyOrderedInKG"] / so["ProductID"].astype(str).map(kgpp)
assert so["Pallet"].notna().all()                                         # mọi sản phẩm đều có KGPerPallet

# (2) Tháng đặt hàng
so["Thang"] = pd.to_datetime(so["OrderedTimestamp"]).dt.to_period("M")    # vd 2025-01
print("Các tháng có trong dữ liệu:", sorted(so["Thang"].astype(str).unique()))

# (3) Gán nhóm khách. Khách không có nhóm (Decision_Log 5.6, ~0,1 tấn) bị bỏ, in ra lượng bị bỏ
so["ShipToGroupID"] = so["ShipToID"].astype(str).map(mp.set_index("ShipToID")["ShipToGroupID"])
khong_nhom = so["ShipToGroupID"].isna()
print(f"Bỏ dòng không có nhóm: {khong_nhom.sum():,} dòng | {so.loc[khong_nhom, 'QtyOrderedInKG'].sum()/1000:,.2f} tấn")
so = so[~khong_nhom]

# (4) Cộng pallet theo nhóm × tháng; bảng ngang: mỗi dòng 1 nhóm, mỗi cột 1 tháng
THANG = pd.period_range("2025-01", "2025-11", freq="M")                   # 11 tháng của kỳ nghiên cứu
thang_nhom = (so.groupby(["ShipToGroupID", "Thang"], observed=True)["Pallet"].sum()
                .unstack("Thang")
                .reindex(columns=THANG)                                   # đủ 11 cột tháng
                .reindex(G)                                               # đủ 3.347 nhóm
                .fillna(0.0))                                             # tháng không đặt hàng = 0
thang_nhom.columns = thang_nhom.columns.astype(str)                       # tên cột "2025-01"... cho parquet
thang_nhom.index.name = "ShipToGroupID"

# (5) Tham số robust
demand = pd.DataFrame({
    "d_bar": thang_nhom.mean(axis=1),                                     # cầu danh nghĩa: trung bình pallet/tháng
    "d_hat": thang_nhom.max(axis=1) - thang_nhom.mean(axis=1),            # độ lệch tối đa: max tháng − trung bình
    "ThangCoCau": (thang_nhom > 0).sum(axis=1),                           # số tháng có đặt hàng
}).reset_index()

# (6) Tóm tắt
print(f"\nTổng d̄ = {demand['d_bar'].sum():,.0f} pallet/tháng | tổng d̂ = {demand['d_hat'].sum():,.0f} pallet/tháng")
print("Tổng pallet theo tháng (cả nước):")
print(thang_nhom.sum().round(0).to_string())
print("\nPhân bố d̄ (pallet/tháng) và tỷ lệ d̂/d̄:")
print(pd.DataFrame({"d_bar": demand["d_bar"],
                    "d_hat/d_bar": demand["d_hat"] / demand["d_bar"]}).describe(
                    percentiles=[.25, .5, .75, .95]).round(2))
print("\nSố tháng có đặt hàng (số tháng: số nhóm):", demand["ThangCoCau"].value_counts().sort_index().to_dict())

# (7) Kiểm tra + xuất
assert len(demand) == len(G) and demand["ShipToGroupID"].is_unique       # đủ 3.347 nhóm, không trùng
assert demand[["d_bar", "d_hat"]].notna().all().all()                     # không ô trống
assert (demand["d_bar"] > 0).all() and (demand["d_hat"] >= 0).all()      # mọi nhóm có cầu; d̂ không âm
demand.to_parquet(OUT / "demand_robust.parquet", index=False)
thang_nhom.reset_index().to_parquet(OUT / "demand_monthly.parquet", index=False)
print(f"\nĐã lưu demand_robust.parquet ({len(demand):,} nhóm) và demand_monthly.parquet (nhóm × 11 tháng)")

## 1.2 Mức độ bất định của cầu (đặc tính ngành personal care / FMCG)

Bằng chứng số cho việc dùng robust: mùa vụ (Ramadan 2025 ≈ tháng 3, Lebaran 31/3), biến động theo nhóm, phân tán khách hàng, cơ cấu SKU, mật độ hàng.

In [ ]:
# ===== 1.2 ĐẶC TÍNH CẦU ĐO TỪ DỮ LIỆU =====
# (1) Mùa vụ: chỉ số tháng = pallet tháng ÷ trung bình 11 tháng
tong_thang = thang_nhom.sum()
print("(1) Chỉ số mùa vụ (1,00 = tháng trung bình):")
print((tong_thang / tong_thang.mean()).round(2).to_string())

# (2) Biến động cầu theo nhóm: hệ số biến thiên CV = độ lệch chuẩn ÷ trung bình qua 11 tháng
cv = thang_nhom.std(axis=1) / thang_nhom.mean(axis=1)
w = thang_nhom.mean(axis=1)                                               # trọng số = cầu trung bình của nhóm
print(f"\n(2) CV theo nhóm: trung vị {cv.median():.2f} | bình quân gia quyền theo cầu {(cv * w).sum() / w.sum():.2f}")
for nguong in [0.25, 0.5, 1.0]:
    print(f"    % cầu nằm ở nhóm có CV > {nguong}: {w[cv > nguong].sum() / w.sum() * 100:.1f}%")

# (3) Phân tán khách hàng: cầu tập trung ở bao nhiêu nhóm
w_sorted = w.sort_values(ascending=False)
for p in [0.05, 0.2, 0.5]:
    n = int(len(w_sorted) * p)
    print(f"(3) Top {p:.0%} nhóm ({n:,} nhóm) chiếm {w_sorted.iloc[:n].sum() / w.sum() * 100:.1f}% cầu")

# (4) Sản phẩm và kích thước dòng đơn (dùng lại bảng so đã đọc ở 1.1)
kg_sp = so.groupby("ProductID", observed=True)["QtyOrderedInKG"].sum().sort_values(ascending=False)
print(f"\n(4) Số SKU có bán B2B: {len(kg_sp):,} | số ship-to có đặt hàng: {so['ShipToID'].nunique():,}")
print(f"    Top 20% SKU chiếm {kg_sp.iloc[:int(len(kg_sp) * 0.2)].sum() / kg_sp.sum() * 100:.1f}% kg")
print("    Kg mỗi dòng đơn (phân vị):", so["QtyOrderedInKG"].quantile([.25, .5, .75, .95]).round(1).to_dict())

# (5) Mật độ hàng: kg/pallet bình quân gia quyền theo kg bán (cũng là hệ số đổi pallet -> tấn cho f3)
kgpp_bq = so["QtyOrderedInKG"].sum() / so["Pallet"].sum()
print(f"\n(5) KGPerPallet: trung vị theo SKU {kgpp.median():,.0f} | bình quân gia quyền theo kg bán {kgpp_bq:,.0f} kg/pallet")
del so                                                                    # giải phóng RAM